In [25]:
# pylint: disable=wrong-import-position
# pylint: disable=wrong-import-order
# pylint: disable=invalid-name
# pylint: disable=ungrouped-imports

"""Demo smart home agent built with polycog cognition."""

## critical for running the tutorial on jupyter notebook
## ignore if running on terminal
import nest_asyncio2  # type: ignore[import-untyped]

nest_asyncio2.apply()

# Human-Cogent Communication
## Translating Natural Language to Cognitive Intent

In previous chapters, we built an agent that operates autonomously using environmental sensors/actuators and a deterministic decision process. Now, we expand our cogent to interact with humans through natural language. We will build this functionality with large language models (LLMs). 

Unlike design patterns like ReACT that provide all information to a language model as soft-structured context and let it arrive at a decision, `cognition` uses LLMs as **intent interpreters**. The LLM translates informal, ambiguous human speech into structured, strongly typed domain contracts that our decision process can evaluate deterministically. 


## Prerequisites: Load Basic Cogent

We begin by loading the baseline smart home cogent implementation built in Tutorial 2 from [smart_home_assistant_v1.py](smart_home_assistant_v1.py). This basic implementation contains device state definitions, physical actuators, sensors, and the core agent state .

In [26]:
## %load smart_home_assistant_v1.py
import os
import time
from dataclasses import dataclass
from enum import Enum, StrEnum, auto
from typing import Any, cast

from cognition import (
    Actuator,
    Cogent,
    DecisionProcess,
    DecisionProcessErrorMessage,
    IOContainer,
    Operator,
    Sensor,
)

from smart_home.client import SmartHomeClient

# pylint: disable=redefined-outer-name


# =============================================================================
# Enums & Data Models
# =============================================================================


class DeviceState(StrEnum):
    """Smart home device operational states."""

    ON = "on"
    OFF = "off"


class ControlSignal(StrEnum):
    """Control commands sent to smart home devices."""

    TURN_ON = "turn_on"
    TURN_OFF = "turn_off"


class Mode(Enum):
    """Operating modes for the home assistant."""

    DAY = auto()
    EVENING = auto()
    MIDNIGHT = auto()


@dataclass(frozen=True)
class Observation:
    """State observation from the home simulation."""

    lamp: DeviceState


@dataclass(frozen=True)
class Action:
    """Command action to execute on home devices."""

    lamp: ControlSignal


@dataclass
class HomeState:
    """Internal state maintained by the cogent"""

    mode: Mode
    timer_expires_at: float = 0.0


# =============================================================================
# Global Client
# =============================================================================

client = SmartHomeClient()
try:
    client.connect()
    print("Successfully connected to Smart Home Simulation!")
except ConnectionError as exc:
    raise SystemExit(
        "Connection failed. Ensure the simulation app is running in another terminal.\n"
        f"Details: {exc}"
    ) from exc


# =============================================================================
# Sensors & Actuators
# =============================================================================


class SmartHomeDevices(Sensor[Observation], Actuator[Action, None]):
    """Sensor and actuator interface for physical home devices."""

    @property
    def name(self) -> str:
        return "devices"

    def sense(self) -> Observation:
        """Fetch current states of smart home devices."""
        response = client.observe()
        return Observation(lamp=response["lamp"])

    def actuate(self, param: Action) -> None:
        """Execute device commands in the simulation."""
        client.actuate("lamp", param.lamp)


# =============================================================================
# Operators
# =============================================================================


class StartTimer(Operator[HomeState]):
    """
    WHEN: Lamp is ON in MIDNIGHT mode and timer is not set.
    THEN: Start a 5-second countdown timer.
    """

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        """Check if mode is MIDNIGHT, lamp is ON, and no timer is running."""
        observation = cast(Observation, io.i.devices)
        return (
            state.mode is Mode.MIDNIGHT
            and DeviceState(observation.lamp) == DeviceState.ON
            and state.timer_expires_at == 0.0
        )

    def perform(self, state: HomeState, io: IOContainer) -> None:
        """Start a 5-second timer."""
        state.timer_expires_at = time.monotonic() + 5.0
        print("--> [Timer Started] 5-second countdown initialized.")


class TurnLightOff(Operator[HomeState]):
    """
    WHEN: Lamp is ON in MIDNIGHT mode and active timer has expired.
    THEN: Turn off the lamp and reset the timer state.
    """

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        """Check if mode is MIDNIGHT, lamp is ON, and timer has expired."""
        observation = cast(Observation, io.i.devices)
        return (
            state.mode is Mode.MIDNIGHT
            and DeviceState(observation.lamp) == DeviceState.ON
            and state.timer_expires_at != 0
            and time.monotonic() >= state.timer_expires_at
        )

    def perform(self, state: HomeState, io: IOContainer) -> None:
        """Turn off the lamp and reset the timer state."""
        io.o.devices(Action(lamp=ControlSignal.TURN_OFF))
        state.timer_expires_at = 0.0
        print("--> [Action Triggered] Timer expired. Turning light off.")


class ResetTimer(Operator[HomeState]):
    """
    WHEN: Lamp is OFF while a timer is active.
    THEN: Reset the timer to zero.
    """

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        """Check if lamp is OFF while a timer is active."""
        observation = cast(Observation, io.i.devices)
        return observation.lamp == DeviceState.OFF and state.timer_expires_at != 0.0

    def perform(self, state: HomeState, io: IOContainer) -> None:
        """Reset the active timer."""
        state.timer_expires_at = 0.0
        print("--> [Internal Action Triggered] Light off, resetting timer.")


# =============================================================================
# Agent run
# =============================================================================


def run_cogent(cogent: Cogent[DecisionProcess[Any]]):
    """Run the cogent's perpetual Perceive-Decide-Act loop."""

    def run_forever(_cogent) -> bool:
        """Keep the execution loop running continuously."""
        return True

    def gate_no_potential_actions(err: DecisionProcessErrorMessage, _cogent) -> bool:
        """
        Handle decision process errors.
        Allows the agent to remain idle when no actions are proposed, while pausing
        briefly to prevent tight CPU looping.
        """
        time.sleep(0.5)
        if err is DecisionProcessErrorMessage.NO_PROPOSAL:
            return (
                True  # Suppress 'no potential action' error and continue execution loop
            )
        raise RuntimeError(err)  # Re-raise unexpected critical errors

    print(
        "Cogent running. Click 'Interrupt Kernel' in Jupyter or press Ctrl+C in terminal to stop.\n"
    )
    try:
        # Pass the loop predicate and error handling policy to the Cogent instance
        cogent(run_forever, dp_err_p=gate_no_potential_actions)
    except KeyboardInterrupt:
        print("\nExecution interrupted.")

Successfully connected to Smart Home Simulation!


## Step 1: Define Human Iteraction Data Contracts and Sensor/Actuator Pair
Just as we built the `SmartHomeDevices` interface to exchange physical telemetry via `IOContainer`, we need a dedicated I/O interface to handle conversational channels.

To establish two-way communication between the resident and the cogent, we define two core components:

- `Utterance` data contract: An immutable `@dataclass(frozen=True)` that encapsulates natural language text phrases passed across the interaction boundary.

- `HumanInteraction` class: A composite interface implementing both Sensor[Utterance] and Actuator[Utterance, None]. It polls the chat API for incoming user messages (`sense`) and dispatches conversational replies back to the resident (`actuate`).


Below, we define the `Utterance` payload contract and the `HumanInteraction` composite interface registered under the lookup key `"interaction"`:

In [27]:
@dataclass(frozen=True)
class Utterance:
    """Natural language message phrase."""

    phrase: str | None


class HumanInteraction(Sensor[Utterance | None], Actuator[Utterance, None]):
    """Sensor and actuator interface for user chat interactions."""

    @property
    def name(self) -> str:
        return "interaction"

    def sense(self) -> Utterance | None:
        """Fetch the latest message from the human."""
        message = client.get_last_message()
        if message:
            return Utterance(phrase=message["text"])
        return Utterance(phrase=None)

    def actuate(self, param: Utterance) -> None:
        """Send a response to the human."""
        client.acknowledge_message()
        if param.phrase:
            client.send_message(param.phrase)

## Step 2: Define Intent Categories
The central challenge of human-agent interaction is translating ambiguous natural language into structured intents that an agent can execute safely. 

In `cognition`, intent taxonomies are defined using `AutoDocEnum` alongside `EnumClassifier`. Each enum member represents a discrete user intent, while its string value provides the semantic context that `cognition` automatically converts into prompt instructions for the underlying language model.

When designing intent categories, keep these tips in mind:

- Model high-level objectives (not low-level APIs): scope intents to the user's conceptual goals (e.g., `TURN_OFF_LAMP`) rather than low-level API mechanics or wire protocols. Downstream decision operators handle the specific API commands and operational state checks required to fulfill the request.

- Provide clear semantic context: write distinct, non-overlapping string definitions for each enum member. Clear, explicit semantic descriptions allow the LLM interpreter to classify incoming text with high precision.

- Decouple understanding from execution: An intent represents what the user asked for, independent of whether the agent will choose to fulfill it. For instance, if a resident requests `TURN_ON_LAMP`, classification succeeds even if a safety rule or schedule policy subsequently prevents the light from turning on.

Below, we define the `ResidentIntent` taxonomy using `AutoDocEnum`. We also define `MyIntent` taxonomy for the cogent that it uses to respond to the human. 

In [28]:
from cognition import AutoDocEnum


class ResidentIntent(AutoDocEnum):
    """Supported resident intent classifications."""

    TURN_OFF_LAMP = "user is asking the assistant to turn the lamp off"
    TURN_ON_LAMP = "user is asking the assistant to turn the lamp on"


class MyIntent(StrEnum):
    """Responses from the assistant to the user"""

    UNKNOWN = "I am sorry; I am not programmed to respond to that."
    CONFIRMATION = "Done."
    GREETING = "Hi!"
    GRATITUDE_ACK = "You are welcome."


## Step 3: Extend State to Capture Intent Processing Stages
To respond to human intent safely, an agent must separate interpretation from execution. Storing the interpretation stage inside the agent's internal `State` establishes a two-phase lifecycle:

- **Interpretation stage**: An interpreter operator parses raw incoming chat text into a classified `ResidentIntent` and records it in `State`.

- **Execution stage**: Downstream decision operators evaluate `state.human_intent` alongside current environmental conditions to deterministically trigger environment actions or dispatch conversational replies.

By persisting both the raw `Utterance` and the classified `ResidentIntent` within State, we decouple intent classification from action execution, ensuring the agent can inspect, evaluate, and respond to active user requests safely.

In [29]:
# Extend State to hold interpretation processing stage
@dataclass
class NewState(HomeState):
    """Internal state to track the stage of intent processing"""

    human_utterance: Utterance | None = None
    human_intent: ResidentIntent | None = None

## Step 4: Configure LLM Provider with `pydantic_ai`

To reliably parse `ResidentIntent` without relying on fragile regular expressions or unvalidated LLM output, `cognition` integrates with **`pydantic_ai`** under the hood.

### What is `pydantic_ai`?
**`pydantic_ai`** is a Python framework designed for building structured, type-safe AI systems:

* **Guaranteed structured output:** leverages Pydantic validation to guarantee that LLM outputs conform strictly to target Python types or Enums (`ResidentIntent`).
* **Provider abstraction:** offers unified interfaces across model providers (OpenAI, Anthropic, Gemini, local models, etc.) under a single standard API.
* **Integration with `cognition`:** `EnumClassifier` class uses `pydantic_ai` models to ensure freeform text translates directly into validated `ResidentIntent` instances without runtime schema errors.

In [30]:
# Install a relevant pydantic-ai module
#!pip install "pydantic-ai-slim[openai]"

from pydantic_ai.models import infer_model

# Configure the LLM provider
# Tutorial uses OpenAI's apis (https://developers.openai.com/api/docs/quickstart)
# Set the OPENAI_API_KEY in your environment (https://help.openai.com/en/articles/5112595-best-practices-for-api-key-safety)
assert "OPENAI_API_KEY" in os.environ, "Environment variable OPENAI_API_KEY is not set."
llm_model = infer_model("openai:gpt-4o")

## Step 5: Implement Intent Interpretation Operator

In this step, we implement `InterpretHumanIntent`: the operator responsible for translating raw text utterances into structured, type-safe cognitive intents.

Under the hood, `cognition`'s `EnumClassifier` serves as an abstraction built over `pydantic_ai`. It automatically constructs a structured classification prompt derived directly from the `ResidentIntent` (`AutoDocEnum`) definition, optimizing LLM inference to reliably map user utterances to one of the enum members.

During execution, the operator reads sensory input from `io.i.interaction`, routes the text phrase through `EnumClassifier`, and populates `state.human_intent`.

Notice how unhandled requests are managed: if an incoming message does not match any valid category defined in `ResidentIntent`, the classifier returns `None`. Rather than attempting an unscripted or hallucinated action, the cogent explicitly notifies the user that it cannot assist. This design enforces a whitelist approach to agent safety. [Read about our architecture commitments and their impact on agent behavior](architecture_design.md).

In [31]:
from cognition import EnumClassifier


class InterpretHumanIntent(Operator[NewState]):
    """
    WHEN: User sends a message via the interaction sensor.
    THEN: Parse the message into a resident intent classification.
    """

    def __init__(self, name: str, **kwargs: Any) -> None:
        super().__init__(name, **kwargs)
        self._interpreter = EnumClassifier(
            enum_type=ResidentIntent,
            task_desc="what is the resident asking the assistant to do",
        )

    def can_perform(self, state: NewState, io: IOContainer) -> bool:
        """Check for unhandled user messages on the interaction link."""
        utterance = cast(Utterance, io.i.interaction)
        if utterance.phrase is not None:
            state.human_utterance = utterance
        return bool(state.human_utterance and not state.human_intent)

    def perform(self, state: NewState, io: IOContainer) -> None:
        """Parse user phrase into a resident intent."""
        print("--> [Interpret Intent triggered]")
        assert (
            state.human_utterance is not None
            and state.human_utterance.phrase is not None
        ), "Expect can_perform to populate utterance"
        state.human_intent = self._interpreter(
            state.human_utterance.phrase, llm=llm_model, num_trials=1
        )[0]
        print(f"--> [Interpreted Intent] human expressed {state.human_intent}")
        if state.human_intent is None:
            io.o.interaction(Utterance(phrase=MyIntent.UNKNOWN))
            state.human_utterance = None

## Step 6: Implement Execution Operators

Next, we implement the action operators (`ActOnHumanIntentTurnOff` and `ActOnHumanIntentTurnOn`) to handle the execution phase of our natural language interaction pipeline.

These operators evaluate `state.human_intent` populated in the previous step against target classifications. When a match occurs, the corresponding operator dispatches control signals to the physical environment, issues conversational feedback to the resident, and resets the active intent state.

This design enforces a strict separation between interpretation and execution. Te language model acts purely as an intent classifier. Explicit, deterministic Python operators hold sole authority over triggering real-world actions. By keeping execution deterministic, you ensure that safety constraints, state checks, and payload construction are entirely predictable and testable. This design helps create a [governance envelope](architecture_design.md) within which the agent operates.

In [32]:
class ActOnHumanIntentTurnOff(Operator[NewState]):
    """
    WHEN: Resident intent is TURN_OFF_LAMP
    THEN: turn the lamp off
    """

    def can_perform(self, state: NewState, io: IOContainer) -> bool:
        return state.human_intent is ResidentIntent.TURN_OFF_LAMP

    def perform(self, state: NewState, io: IOContainer) -> None:
        io.o.devices(Action(lamp=ControlSignal.TURN_OFF))
        io.o.interaction(Utterance(phrase=MyIntent.CONFIRMATION))

        print("--> [Action Triggered] Reacting to human request. Turning light OFF.")
        state.human_intent = None
        state.human_utterance = None


class ActOnHumanIntentTurnOn(Operator[NewState]):
    """
    WHEN: Resident intent is TURN_ON_LAMP
    THEN: Turn the lamp off
    """

    def can_perform(self, state: NewState, io: IOContainer) -> bool:
        return state.human_intent is ResidentIntent.TURN_ON_LAMP

    def perform(self, state: NewState, io: IOContainer) -> None:
        io.o.devices(Action(lamp=ControlSignal.TURN_ON))
        io.o.interaction(Utterance(phrase=MyIntent.CONFIRMATION))
        print("--> [Action Triggered] Reacting to human request. Turning light ON.")
        state.human_intent = None
        state.human_utterance = None

## Step 7: Launch the Conversational Cogent

Our `assistant` cogent is fully configured. We can now launch the continuous execution loop directly. Try asking the assistant to turn the lamp on or off in the simulation TUI.

In [33]:
state = NewState(mode=Mode.MIDNIGHT)
dp = DecisionProcess[NewState](lambda: state)

assistant: Cogent[DecisionProcess[NewState]] = Cogent(decision_process=dp)
devices = SmartHomeDevices()
interaction = HumanInteraction()
assistant.add_sensor(devices).add_actuator(devices)
assistant.add_sensor(interaction).add_actuator(interaction)

## autonomous action operators / bound to older State, but work
assistant.dp.add_operator(TurnLightOff("turn_light_off", terminal=True))  # type: ignore[arg-type]
assistant.dp.add_operator(StartTimer("start_timer"))  # type: ignore[arg-type]
assistant.dp.add_operator(ResetTimer("reset_timer"))  # type: ignore[arg-type]

## intent-interpretation and intent-driven-action operators
assistant.dp.add_operator(
    InterpretHumanIntent("interpret_human_response", terminal=True)
)
assistant.dp.add_operator(ActOnHumanIntentTurnOn("intent_turn_light_on", terminal=True))
assistant.dp.add_operator(
    ActOnHumanIntentTurnOff("intent_turn_light_off", terminal=True)
)

(<cognition.util.misc.StringifiedFunction at 0x1074d5230>,
 <cognition.decision.dp.DecisionProcess at 0x11607ac10>)

In [ ]:
run_cogent(assistant)

Cogent running. Click 'Interrupt Kernel' in Jupyter or press Ctrl+C in terminal to stop.

--> [Timer Started] 5-second countdown initialized.
--> [Action Triggered] Timer expired. Turning light off.
--> [Interpret Intent triggered]
--> [Interpreted Intent] human expressed ResidentIntent.TURN_ON_LAMP
--> [Action Triggered] Reacting to human request. Turning light ON.
--> [Timer Started] 5-second countdown initialized.
--> [Action Triggered] Timer expired. Turning light off.


## 🛠️ Developer Exercise: Extend the Cogent to Change Operating Modes

Currently, the cogent defaults to `MIDNIGHT` mode, so turning on the lamp always triggers the 5-second auto-off timer.

**Your Goal:** Allow the user to change the operational mode via natural language (e.g., *"Set home mode to Day"* or *"Switch to Evening mode"*).

### Tasks to Complete:
1. **Update `ResidentIntent`:** Add a new intent classification `SET_MODE` with a descriptive string for `AutoDocEnum` (e.g., `"user is asking to change or set the operational mode"`).
2. **Implement `ActOnHumanIntentSetMode` Operator:**
   * Guard condition (`can_perform`): Check if `state.human_intent` is `ResidentIntent.SET_MODE`.
   * Action (`perform`): Update `state.mode` (e.g., to `Mode.DAY`), send a confirmation message back to the resident via `io.o.interaction`, and clear `state.human_intent`.
3. **Register the Operator:** Add your new operator to `assistant.dp`.
4. **Test in the TUI:** Ask the assistant to switch to Day mode, then turn the lamp on. Verify that the light now stays on indefinitely because the midnight timer rule no longer applies!

Check out the full implementation at [smart_home_assistant_v3.py](smart_home_assistant_v3.py).